In [1]:
!pip install -q datasets spacy huggingface_hub
!python -m spacy download en_core_web_sm -q
!python -m spacy download de_core_news_sm -q
print("Package installed")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 38.7 MB/s eta 0:00:0000:010:01
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.6/14.6 MB 68.2 MB/s eta 0:00:0000:0100:01
✔ Download and installation successful
You can now load the package via spacy.load('de_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
Package installed


In [2]:
from datasets import load_dataset

dataset = load_dataset('wmt/wmt14','de-en')

train_data = dataset["train"].select(range(500000))
val_data   = dataset["validation"]
test_data  = dataset["test"]

print('Length of train data : ',len(train_data))

README.md: 0.00B [00:00, ?B/s]

de-en/train-00000-of-00003.parquet:   0%|          | 0.00/280M [00:00<?, ?B/s]

de-en/train-00001-of-00003.parquet:   0%|          | 0.00/265M [00:00<?, ?B/s]

de-en/train-00002-of-00003.parquet:   0%|          | 0.00/273M [00:00<?, ?B/s]

de-en/validation-00000-of-00001.parquet:   0%|          | 0.00/474k [00:00<?, ?B/s]

de-en/test-00000-of-00001.parquet:   0%|          | 0.00/509k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/4508785 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/3003 [00:00<?, ? examples/s]

Length of train data :  500000


In [3]:
import spacy
from collections import Counter

nlp_en = spacy.load("en_core_web_sm")
nlp_de = spacy.load("de_core_news_sm")

def tokenize_en(text): return [t.text.lower() for t in nlp_en.tokenizer(text)]
def tokenize_de(text): return [t.text.lower() for t in nlp_de.tokenizer(text)]

PAD, UNK, BOS, EOS = 0, 1, 2, 3
def build_vocab(sentences,tokenizer,min_freq = 2):
  cnt = Counter()
  for sent in sentences:
    cnt.update(tokenizer(sent))
  vocab = {'PAD':0,'UNK':1,'BOS':2,'EOS':3}
  for i,j in cnt.items():
    if j > min_freq:
      vocab[i] = len(vocab)
  return vocab

vocab_en = build_vocab([x["translation"]['en'] for x in train_data],tokenize_en)
vocab_de = build_vocab([x["translation"]['de'] for x in train_data],tokenize_de)
inv_vocab_de = {v: k for k, v in vocab_de.items()}

print("English vocab:", len(vocab_en))
print("German  vocab:", len(vocab_de))

English vocab: 29115
German  vocab: 67036


In [4]:
import pickle

with open("vocab_en.pkl", "wb") as f:
    pickle.dump(vocab_en, f)

with open("vocab_de.pkl", "wb") as f:
    pickle.dump(vocab_de, f)

In [5]:
import pickle

with open("vocab_en.pkl", "rb") as f:
    vocab_en = pickle.load(f)

with open("vocab_de.pkl", "rb") as f:
    vocab_de = pickle.load(f)

In [32]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

def encode(sentence,vocab,tokenizer):
  token = [BOS] + [vocab.get(x,UNK) for x in tokenizer(sentence)] + [EOS]
  return torch.tensor(token,dtype=torch.long)

def decode(tensor,inv_voc):
  return ' '.join(inv_voc.get(x,'UNK') for x in tensor.tolist() if x not in (PAD,BOS,EOS))

In [7]:
class TranslationDataset(Dataset):
    def __init__(self, data):
        self.data = data

    def __len__(self):
        return len(self.data)

    def __getitem__(self, i):
        item = self.data[i]["translation"]

        return (
            encode(item["en"], vocab_en, tokenize_en),
            encode(item["de"], vocab_de, tokenize_de)
        )

def collate_fn(batch):
    src, tgt = zip(*batch)
    src = pad_sequence(src, batch_first=True, padding_value=PAD)
    tgt = pad_sequence(tgt, batch_first=True, padding_value=PAD)
    return src, tgt

train_loader = DataLoader(TranslationDataset(train_data), batch_size=32, shuffle=True,  collate_fn=collate_fn)
val_loader   = DataLoader(TranslationDataset(val_data),   batch_size=32, shuffle=False, collate_fn=collate_fn)

src, tgt = next(iter(train_loader))
print("src shape:", src.shape)
print("tgt shape:", tgt.shape)

src shape: torch.Size([32, 59])
tgt shape: torch.Size([32, 59])


In [8]:
import math

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, dropout=0.1, max_len=5000):
        super().__init__()
        self.dropout = nn.Dropout(dropout)

        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2) * -(math.log(10000.0) / d_model)
        )
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x):
        x = x + self.pe[:, :x.size(1)]
        return self.dropout(x)

In [9]:
class TranslatorModel(nn.Module):
  def __init__(self,src_vocab,tgt_vocab,d_model = 256,nhead = 8,num_encoder_layers = 3,num_decoder_layers = 3,dim_feedforward = 512,dropout = 0.1):
    super().__init__()
    self.src_emb = nn.Embedding(src_vocab,d_model,padding_idx=0)
    self.tgt_emb = nn.Embedding(tgt_vocab,d_model,padding_idx=0)
    self.pos_enc = PositionalEncoding(d_model,dropout)
    self.transformer = nn.Transformer(d_model,nhead,num_encoder_layers,num_decoder_layers,dim_feedforward,dropout,batch_first=True)
    self.fc = nn.Linear(d_model,tgt_vocab)
    self.d_model = d_model

  def forward(self,src,tgt,src_key_padding_mask,tgt_key_padding_mask,tgt_mask):
    src = self.src_emb(src) * math.sqrt(self.d_model)
    tgt = self.tgt_emb(tgt) * math.sqrt(self.d_model)
    out = self.transformer(src,tgt,tgt_mask = tgt_mask,src_key_padding_mask = src_key_padding_mask,tgt_key_padding_mask = tgt_key_padding_mask)
    return self.fc(out)

device = "cuda" if torch.cuda.is_available() else "cpu"
print('Device',device)
model = TranslatorModel(len(vocab_en),len(vocab_de)).to(device)
params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model ready! Parameters: {params:,}")

Device cuda
Model ready! Parameters: 45,797,596


In [10]:
def make_masks(src, tgt):
    src_pad_mask = (src == PAD).to(torch.bool)
    tgt_pad_mask = (tgt == PAD).to(torch.bool)

    tgt_len = tgt.size(1)
    tgt_mask = nn.Transformer.generate_square_subsequent_mask(tgt_len, device=tgt.device)

    return src_pad_mask, tgt_pad_mask, tgt_mask

In [11]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(),lr = 0.0001)

def train_one_epoch(train_loader):
  model.train()
  total_loss = 0
  for ind,(src,tgt) in enumerate(train_loader):
    optimizer.zero_grad()
    src,tgt = src.to(device),tgt.to(device)
    tgt_in = tgt[:,:-1]
    tgt_out = tgt[:,1:]
    src_pad_mask,tgt_pad_mask,tgt_mask = make_masks(src,tgt_in)
    out = model(src,tgt_in,src_pad_mask,tgt_pad_mask,tgt_mask)
    loss = criterion(out.reshape(-1,out.shape[-1]),tgt_out.reshape(-1))
    loss.backward()
    optimizer.step()
    total_loss += loss.item()
  return total_loss/len(train_loader)

def evaluate(loader):
    model.eval()
    total_loss = 0
    with torch.no_grad():
        for src, tgt in loader:
            src, tgt = src.to(device), tgt.to(device)
            tgt_in  = tgt[:, :-1]
            tgt_out = tgt[:, 1:]
            src_pad, tgt_pad, tgt_mask = make_masks(src, tgt_in)
            out  = model(src, tgt_in, src_pad, tgt_pad, tgt_mask)
            B, T, V = out.shape
            loss = criterion(out.reshape(B*T, V), tgt_out.reshape(-1))
            total_loss += loss.item()
    return total_loss / len(loader)

EPOCHS = 5
best_val = float("inf")
print("Starting training...\n")
for epoch in range(1, EPOCHS+1):
    tr  = train_one_epoch(train_loader)
    val = evaluate(val_loader)
    saved = ""
    if val < best_val:
        best_val = val
        torch.save(model.state_dict(), "best_model.pt")
        saved = "  ← saved"
    print(f"Epoch {epoch:2d}/{EPOCHS}  train: {tr:.4f}  val: {val:.4f}{saved}")
print("\nDone!")

Starting training...



/usr/local/lib/python3.12/dist-packages/torch/nn/modules/activation.py:1336: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = F._canonical_mask(
/usr/local/lib/python3.12/dist-packages/torch/nn/modules/transformer.py:531: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. We recommend specifying layout=torch.jagged when constructing a nested tensor, as this layout receives active development, has better operator coverage, and works with torch.compile. (Triggered internally at /pytorch/aten/src/ATen/NestedTensorImpl.cpp:178.)
  output = torch._nested_tensor_from_mask(


Epoch  1/5  train: 2.0463  val: 2.3831  ← saved
Epoch  2/5  train: 1.6684  val: 2.2271  ← saved
Epoch  3/5  train: 1.5247  val: 2.1297  ← saved
Epoch  4/5  train: 1.4371  val: 2.0599  ← saved
Epoch  5/5  train: 1.3707  val: 2.0179  ← saved

Done!


In [30]:
model.load_state_dict(torch.load("best_model.pt", map_location=device))
model.eval()

def translate(sentence, max_len=50):
    src = encode(
        sentence,
        vocab_en,
        tokenize_en
    ).unsqueeze(0).to(device)

    generated = [BOS]

    with torch.no_grad():

        for _ in range(max_len):

            tgt = torch.tensor(
                generated,
                dtype=torch.long,
                device=device
            ).unsqueeze(0)

            src_pad_mask, tgt_pad_mask, tgt_mask = make_masks(
                src,
                tgt
            )

            output = model(
                src,
                tgt,
                src_pad_mask,
                tgt_pad_mask,
                tgt_mask
            )

            next_token = output[:, -1, :].argmax(-1).item()

            generated.append(next_token)

            if next_token == EOS:
                break
    return decode(torch.tensor(generated),inv_vocab_de)

In [35]:
translate('I am good')

'ich bin sicher .'